# 体積ハカルくん - Google Colab試作版

iPhoneで対象物の周囲を撮影した動画から、**視体積（visual hull）**を作り、体積を `cm³` / `mL` で推定します。

## この試作版の対象

- 静止している、不透明な小物
- 深い凹みが少ない形状
- A4マーカーボード内に収まる対象
- 背景と色・輪郭を区別しやすい対象

> **重要:** 見えない凹みは埋まった形として計算されるため、真の体積より大きくなることがあります。精密測定、診断、安全性・品質などの重要な判断には使用しないでください。


## 撮影前の準備

1. [A4マーカーボードPDF](https://satorumuro.github.io/hakarukun-web/volume/volume-marker-board-a4.pdf)を**100%・実際のサイズ**で印刷します。
2. 下部の確認線が正確に100 mmであることを定規で確認します。
3. 対象物をボード中央に置きます。
4. ズームを変えず、ボードをなるべく画面に残したまま、10〜20秒かけて一周撮影します。
5. 影、反射、手ぶれを避け、対象の高さが分かるよう少し斜め上から撮影します。


In [ ]:
# 1. 必要なライブラリを準備します（初回は数分かかります）
%pip install -q "numpy<2.3" "rembg==2.0.67" "onnxruntime==1.22.1" "trimesh==4.8.1" "scikit-image==0.25.2" "plotly==6.3.0"
%pip uninstall -q -y opencv-python-headless
%pip install -q --force-reinstall --no-deps "opencv-contrib-python-headless==4.12.0.88"


In [ ]:
# 2. 体積計算パイプラインを読み込みます
from pathlib import Path
import urllib.request

module_path = Path('/content/volume_pipeline.py')
module_urls = [
    'https://raw.githubusercontent.com/SatoruMuro/hakarukun-web/main/colab/volume_pipeline.py',
    'https://raw.githubusercontent.com/SatoruMuro/hakarukun-web/agent/volume-colab-prototype/colab/volume_pipeline.py',
]
for url in module_urls:
    try:
        urllib.request.urlretrieve(url, module_path)
        if module_path.stat().st_size > 5000:
            print('Loaded:', url)
            break
    except Exception:
        continue
else:
    raise RuntimeError('体積計算モジュールを取得できませんでした。しばらくしてから再実行してください。')

from volume_pipeline import (
    BoardSpec, VolumeBounds, calibrate_camera_from_board, carve_visual_hull,
    estimate_board_poses, extract_video_frames, save_contact_sheet,
    save_volume_outputs, segment_pose_frames_with_rembg,
)
print('準備できました。')


In [ ]:
# 3. iPhoneで撮影した動画を選びます
from google.colab import files
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('動画が選択されていません。')
video_name = next(iter(uploaded))
video_path = Path('/content') / video_name
print('動画:', video_path.name, f'({video_path.stat().st_size / 1024 / 1024:.1f} MB)')


In [ ]:
# 4. 測定条件を設定します
TARGET_FRAMES = 32 # @param {type:"slider", min:16, max:60, step:4}
MAX_OBJECT_HEIGHT_MM = 150 # @param {type:"slider", min:30, max:220, step:5}
BOARD_MARGIN_MM = 25 # @param {type:"slider", min:10, max:50, step:5}
VOXEL_SIZE_MM = 4 # @param {type:"slider", min:2, max:8, step:1}
SUPPORT_RATIO = 0.72 # @param {type:"slider", min:0.55, max:0.95, step:0.01}

spec = BoardSpec()
bounds = VolumeBounds(
    x_min_m=BOARD_MARGIN_MM / 1000,
    x_max_m=spec.width_m - BOARD_MARGIN_MM / 1000,
    y_min_m=BOARD_MARGIN_MM / 1000,
    y_max_m=spec.height_m - BOARD_MARGIN_MM / 1000,
    height_m=MAX_OBJECT_HEIGHT_MM / 1000,
)
bounds.validate(spec)
print(f'測定範囲: {bounds.x_max_m-bounds.x_min_m:.3f} × {bounds.y_max_m-bounds.y_min_m:.3f} × {bounds.height_m:.3f} m')


In [ ]:
# 5. 動画から手ぶれの少ないフレームを抽出します
from IPython.display import Image, display
work_dir = Path('/content/volume_hakarukun_work')
work_dir.mkdir(exist_ok=True)
frames = extract_video_frames(video_path, work_dir / 'frames', target_frames=TARGET_FRAMES)
sheet = save_contact_sheet(frames, work_dir / 'frames_contact_sheet.jpg')
print(f'{len(frames)}フレームを抽出しました。ボードと対象物が一周を通して見えるか確認してください。')
display(Image(filename=str(sheet), width=960))


In [ ]:
# 6. マーカーボードからカメラ内部パラメータと各フレームの位置を求めます
camera_matrix, distortion, reprojection_error, calibrated_paths = calibrate_camera_from_board(frames, spec)
poses = estimate_board_poses(frames, camera_matrix, distortion, spec)
print(f'利用可能フレーム: {len(poses)} / {len(frames)}')
print(f'再投影誤差: {reprojection_error:.3f} px')
if reprojection_error > 2.0:
    print('警告: カメラ推定誤差が大きめです。手ぶれやボードの折れ・反りを確認してください。')


In [ ]:
# 7. 各方向の対象領域を自動抽出します（初回はモデル取得のため時間がかかります）
segmented = segment_pose_frames_with_rembg(
    poses, work_dir / 'masks', camera_matrix, distortion, bounds
)
mask_sheet = save_contact_sheet(
    [item.image_path for item in segmented],
    work_dir / 'mask_contact_sheet.jpg',
    [item.mask_path for item in segmented],
)
print('緑色が測定対象だけを覆っているか確認してください。')
display(Image(filename=str(mask_sheet), width=960))
print('大きくずれている場合は、背景を単純にして撮り直してください。')


In [ ]:
# 8. 複数方向の輪郭から3D形状を削り出し、体積を計算します
voxel_size_m = VOXEL_SIZE_MM / 1000
minimum_views = max(4, min(8, len(segmented) // 3))
occupancy, axes, hit_count, view_count = carve_visual_hull(
    segmented, camera_matrix, distortion, bounds,
    voxel_size_m=voxel_size_m,
    support_ratio=SUPPORT_RATIO,
    minimum_views=minimum_views,
)
result_dir = Path('/content/volume_hakarukun_result')
result = save_volume_outputs(
    occupancy, axes, voxel_size_m, result_dir, len(segmented), minimum_views, SUPPORT_RATIO
)
print(f'推定体積: {result.voxel_volume_cm3:,.2f} cm³（約 {result.voxel_volume_cm3:,.2f} mL）')
print(f'使用フレーム: {result.usable_frames} / ボクセル寸法: {result.voxel_size_mm} mm')
print('注意:', result.warning)


In [ ]:
# 9. 3Dモデルを表示します
import plotly.graph_objects as go
import trimesh
mesh = trimesh.load(result_dir / 'volume_model.stl', force='mesh')
vertices = mesh.vertices * 1000  # mm
faces = mesh.faces
figure = go.Figure(data=[go.Mesh3d(
    x=vertices[:, 0], y=vertices[:, 1], z=-vertices[:, 2],
    i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
    color='#08775f', opacity=0.82, flatshading=False,
)])
figure.update_layout(
    title=f'推定体積 {result.voxel_volume_cm3:,.2f} cm³',
    scene=dict(aspectmode='data', xaxis_title='X (mm)', yaxis_title='Y (mm)', zaxis_title='高さ (mm)'),
    margin=dict(l=0, r=0, b=0, t=45),
)
figure.show()


In [ ]:
# 10. 結果一式をZIPでダウンロードします
import shutil
archive = shutil.make_archive('/content/volume_hakarukun_result', 'zip', result_dir)
files.download(archive)


## 結果の見方

- `result.json`: 推定体積と測定条件
- `volume_model.glb`: Web表示向け3Dモデル
- `volume_model.stl`: 3D編集・確認向けモデル
- `occupancy_grid.npz`: 再解析用の3D占有データ

同じ対象を3回撮影し、結果のばらつきを確認してください。既知体積の箱や円柱で誤差を評価してから、対象範囲を広げることを推奨します。
